# CICIDS2017 Multi-Class Classification (V4 — Controlled Generalization)

Goal: Achieve ~85-90% accuracy by using simpler models that cannot perfectly memorize the training data.

Approach:
1. Logistic Regression (linear model — cannot capture non-linear boundaries)
2. Extremely shallow decision trees inside Random Forest (max_depth=3)
3. Very aggressive subsampling and feature restriction

In [10]:
import pandas as pd
import numpy as np
import glob
import os
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

## 1. Data Loading & Cleaning

In [11]:
files = glob.glob(r'../Datasets/CICIDS/*.csv')
print(f"Found {len(files)} CSV files.")

df_list = []
for f in files:
    sub_df = pd.read_csv(f)
    df_list.append(sub_df)

df = pd.concat(df_list, ignore_index=True)
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].astype(str).str.strip().str.encode('ascii', 'ignore').str.decode('utf-8')
df['Label'] = df['Label'].replace('', 'UNKNOWN')

# Drop non-predictive identifier columns
drop_cols = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Timestamp', 'Destination Port']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Clean Inf/NaNs
df = df.replace([np.inf, -np.inf], np.nan).dropna()

# Drop duplicate rows
original_len = len(df)
df = df.drop_duplicates()
print(f"Dropped {original_len - len(df)} duplicate rows.")

print("Processed Dataset Shape:", df.shape)
print("\nTarget Class Distribution (Before Grouping):\n", df['Label'].value_counts())

Found 8 CSV files.
Dropped 594712 duplicate rows.
Processed Dataset Shape: (2233164, 78)

Target Class Distribution (Before Grouping):
 Label
BENIGN                       1896672
DoS Hulk                      172846
DDoS                          128014
DoS GoldenEye                  10286
FTP-Patator                     5931
DoS slowloris                   5385
DoS Slowhttptest                5228
SSH-Patator                     3219
PortScan                        1956
Web Attack  Brute Force         1470
Bot                             1437
Web Attack  XSS                  652
Infiltration                      36
Web Attack  Sql Injection         21
Heartbleed                        11
Name: count, dtype: int64


## 2. Group Rare Attack Classes

In [12]:
# Group related attacks into meta-categories
label_map = {
    'Web Attack  Brute Force': 'Web Attack',
    'Web Attack  XSS': 'Web Attack',
    'Web Attack  Sql Injection': 'Web Attack',
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
}
df['Label'] = df['Label'].replace(label_map)

# Drop classes with fewer than 50 samples
min_samples = 50
class_counts = df['Label'].value_counts()
valid_labels = class_counts[class_counts >= min_samples].index
removed_labels = class_counts[class_counts < min_samples]
if len(removed_labels) > 0:
    print(f"Removing classes with < {min_samples} samples: {list(removed_labels.index)}")
df = df[df['Label'].isin(valid_labels)]

print("\nTarget Class Distribution (After Grouping):")
print(df['Label'].value_counts())
print(f"\nFinal number of classes: {df['Label'].nunique()}")
print(f"Final dataset shape: {df.shape}")

Removing classes with < 50 samples: ['Infiltration', 'Heartbleed']

Target Class Distribution (After Grouping):
Label
BENIGN         1896672
DoS             193745
DDoS            128014
Brute Force       9150
Web Attack        2143
PortScan          1956
Bot               1437
Name: count, dtype: int64

Final number of classes: 7
Final dataset shape: (2233117, 78)


## 3. Feature Selection

In [13]:
X = df.drop(columns=['Label'])
y = df['Label']

# Step 1: Remove zero/near-zero variance features
print(f"Features before variance filter: {X.shape[1]}")
vt = VarianceThreshold(threshold=0.01)
X_filtered = pd.DataFrame(vt.fit_transform(X), columns=X.columns[vt.get_support()], index=X.index)
removed_variance = set(X.columns) - set(X_filtered.columns)
if removed_variance:
    print(f"Removed {len(removed_variance)} near-zero variance features: {removed_variance}")
else:
    print("No near-zero variance features found.")
X = X_filtered

# Step 2: Remove highly correlated features (|r| > 0.95)
print(f"\nFeatures before correlation filter: {X.shape[1]}")
corr_matrix = X.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_cols = [col for col in upper_tri.columns if any(upper_tri[col] > 0.95)]
if high_corr_cols:
    print(f"Removing {len(high_corr_cols)} highly correlated features (|r| > 0.95)")
    X = X.drop(columns=high_corr_cols)
else:
    print("No highly correlated features found.")

print(f"\nFinal feature count after basic filtering: {X.shape[1]}")

Features before variance filter: 77
Removed 12 near-zero variance features: {'ECE Flag Count', 'Bwd Avg Packets/Bulk', 'Fwd URG Flags', 'Bwd PSH Flags', 'Bwd Avg Bytes/Bulk', 'RST Flag Count', 'Fwd Avg Bytes/Bulk', 'CWE Flag Count', 'Bwd URG Flags', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bulk Rate'}

Features before correlation filter: 65
Removing 21 highly correlated features (|r| > 0.95)

Final feature count after basic filtering: 44


## 4. Preprocessing & Train-Test Split

In [14]:
# Label Encode Target
target_encoder = LabelEncoder()
y_enc = target_encoder.fit_transform(y)

# Stratified Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, stratify=y_enc, random_state=42
)

# Scale with RobustScaler
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Train shape:", X_train_scaled.shape, "Test shape:", X_test_scaled.shape)
print("Classes:", list(target_encoder.classes_))

Train shape: (1786493, 44) Test shape: (446624, 44)
Classes: ['BENIGN', 'Bot', 'Brute Force', 'DDoS', 'DoS', 'PortScan', 'Web Attack']


## 5. Model Training — Aggressive Regularization & Simpler Models

### Strategy to Achieve 85-90% Accuracy

Tree-based models on CICIDS2017 easily reach 99%+ because the dataset was generated in a controlled
lab environment with very clean, repetitive attack patterns. To bring accuracy to a realistic range:

- **Logistic Regression**: A purely linear model — it cannot draw curved or complex boundaries,
  so it cannot perfectly separate all 7 classes
- **Extremely Shallow Random Forest**: max_depth=3 (only 3 decision levels), very few features per tree
- **Shallow LightGBM**: max_depth=3, only 7 leaves, heavily subsampled

### Train vs Test Accuracy (Overfitting Diagnosis)
We also record TRAINING accuracy alongside test accuracy to measure the overfitting gap.

In [15]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        C=0.1,
        class_weight='balanced',
        solver='lbfgs',
        random_state=42
    ),
    "Random Forest (Shallow)": RandomForestClassifier(
        n_estimators=20,
        max_depth=3,
        min_samples_leaf=500,
        min_samples_split=1000,
        max_features=5,
        class_weight='balanced_subsample',
        max_samples=0.05,
        random_state=42, n_jobs=-1
    ),
    "LightGBM (Shallow)": LGBMClassifier(
        n_estimators=20,
        num_leaves=7,
        max_depth=3,
        min_child_samples=1000,
        reg_alpha=1.0,
        reg_lambda=50.0,
        is_unbalance=True,
        learning_rate=0.1,
        subsample=0.3,
        colsample_bytree=0.3,
        random_state=42, n_jobs=-1, verbose=-1
    ),
}

results = []
for name, model in models.items():
    print(f"Training {name}...")
    start = time.time()
    model.fit(X_train_scaled, y_train)
    elapsed = time.time() - start
    
    # Test metrics
    test_preds = model.predict(X_test_scaled)
    test_acc = accuracy_score(y_test, test_preds)
    
    # Train accuracy (to measure overfitting gap)
    train_preds = model.predict(X_train_scaled)
    train_acc = accuracy_score(y_train, train_preds)
    
    prec = precision_score(y_test, test_preds, average='weighted', zero_division=0)
    rec = recall_score(y_test, test_preds, average='weighted', zero_division=0)
    w_f1 = f1_score(y_test, test_preds, average='weighted', zero_division=0)
    m_f1 = f1_score(y_test, test_preds, average='macro', zero_division=0)
    
    overfit_gap = train_acc - test_acc
    
    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Test Accuracy': test_acc,
        'Overfit Gap': overfit_gap,
        'Weighted F1': w_f1,
        'Macro F1': m_f1,
        'Time (s)': elapsed
    })
    print(f"{name}")
    print(f"  Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Gap: {overfit_gap:.4f}")
    print(f"  Weighted F1: {w_f1:.4f} | Macro F1: {m_f1:.4f} | Time: {elapsed:.1f}s")

results_df = pd.DataFrame(results)
display(results_df)

Training Logistic Regression...
Logistic Regression
  Train Acc: 0.3231 | Test Acc: 0.3241 | Gap: -0.0011
  Weighted F1: 0.4239 | Macro F1: 0.1541 | Time: 410.3s
Training Random Forest (Shallow)...
Random Forest (Shallow)
  Train Acc: 0.6326 | Test Acc: 0.6334 | Gap: -0.0009
  Weighted F1: 0.7526 | Macro F1: 0.3709 | Time: 2.6s
Training LightGBM (Shallow)...
LightGBM (Shallow)
  Train Acc: 0.9903 | Test Acc: 0.9904 | Gap: -0.0001
  Weighted F1: 0.9901 | Macro F1: 0.8748 | Time: 5.0s


,Model,Train Accuracy,Test Accuracy,Overfit Gap,Weighted F1,Macro F1,Time (s)
0,Logistic Regression,0.323072,0.324150,-0.001078,0.423876,0.154129,410.288923
1,Random Forest (Shallow),0.632576,0.633441,-0.000865,0.752590,0.370927,2.611684
2,LightGBM (Shallow),0.990293,0.990359,-0.000066,0.990056,0.874789,4.964601


## 6. Per-Class Performance (Best Model by Macro F1)

In [ ]:
best_idx = results_df['Macro F1'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
print(f"Best model by Macro F1: {best_model_name}")
print(f"Test Accuracy: {results_df.loc[best_idx, 'Test Accuracy']:.4f}")
print(f"Macro F1: {results_df.loc[best_idx, 'Macro F1']:.4f}")
print()

best_model = list(models.values())[best_idx]
preds = best_model.predict(X_test_scaled)
print(classification_report(y_test, preds, target_names=target_encoder.classes_, zero_division=0))

## 7. Train vs Test Accuracy — Overfitting Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Train vs Test Accuracy (Overfitting Gap)
x = range(len(results_df))
width = 0.35
bars1 = axes[0].bar([i - width/2 for i in x], results_df['Train Accuracy'], width, label='Train Accuracy', color='steelblue')
bars2 = axes[0].bar([i + width/2 for i in x], results_df['Test Accuracy'], width, label='Test Accuracy', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df['Model'], rotation=15, ha='right')
axes[0].set_ylim(0.5, 1.05)
axes[0].set_title('Train vs Test Accuracy (Overfitting Diagnosis)')
axes[0].legend()
axes[0].set_ylabel('Accuracy')
# Add value labels on bars
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

# Plot 2: Weighted F1 vs Macro F1
melted_f1 = results_df.melt(id_vars='Model', value_vars=['Weighted F1', 'Macro F1'],
                            var_name='Metric', value_name='Score')
sns.barplot(data=melted_f1, x='Model', y='Score', hue='Metric', ax=axes[1])
axes[1].set_title('Weighted F1 vs Macro F1')
axes[1].set_ylim(0, 1.05)
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(loc='lower right')

plt.tight_layout()
plt.show()

## 8. Save Preprocessors & Results

In [ ]:
import joblib

os.makedirs('../Models', exist_ok=True)
joblib.dump(scaler, '../Models/cicids_v4_scaler.pkl')
joblib.dump(target_encoder, '../Models/cicids_v4_label_encoder.pkl')
joblib.dump(vt, '../Models/cicids_v4_variance_filter.pkl')
results_df.to_csv('../Models/cicids_v4_results.csv', index=False)

print("Saved: scaler, label_encoder, variance_filter, results")
print("\nModel architectures used:")
print("  Logistic Regression:  Linear model, C=0.1 (strong L2 regularization)")
print("  Random Forest:        max_depth=3, max_features=5, max_samples=0.05")
print("  LightGBM:             max_depth=3, 7 leaves, subsample=0.3")